# AutoEvolve-ML: Self-Improving Model Training

This notebook connects to your GitHub repository and runs training sessions that automatically save progress back to GitHub.

## Setup Instructions
1. Run the setup cell to clone the repository
2. Configure your GitHub token (with repo write permissions)
3. Run training cells
4. Weights are automatically pushed back to GitHub

In [ ]:
# Setup: Clone repository
import os
from google.colab import userdata

# Get GitHub token from Colab secrets
# Add your token in Colab: Secrets -> Add Secret -> Name: GITHUB_TOKEN
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except:
    GITHUB_TOKEN = input('Enter your GitHub Personal Access Token: ')

REPO_URL = 'https://github.com/Tommaso-R-Marena/AutoEvolve-ML.git'

# Clone with authentication
!git clone https://{GITHUB_TOKEN}@github.com/Tommaso-R-Marena/AutoEvolve-ML.git
os.chdir('AutoEvolve-ML')

# Configure git
!git config --global user.email "colab@autoevolve.ml"
!git config --global user.name "Colab AutoEvolve"

In [ ]:
# Install dependencies
!pip install -q torch numpy scikit-learn requests matplotlib pandas

In [ ]:
# Pull latest changes
!git pull origin main

In [ ]:
# Run training
!python train.py

In [ ]:
# Push weights back to GitHub
from datetime import datetime

commit_msg = f"Colab training update: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

!git add model_checkpoint.pth metrics.json
!git commit -m "{commit_msg}"
!git push origin main

print('✓ Weights saved to GitHub!')

In [ ]:
# Visualize training progress
import json
import matplotlib.pyplot as plt

with open('metrics.json', 'r') as f:
    history = json.load(f)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Training Loss', marker='o')
plt.plot(history['val_loss'], label='Validation Loss', marker='s')
plt.xlabel('Training Cycle')
plt.ylabel('Loss')
plt.title('Model Performance Over Time')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history['complexity'], marker='o', color='green')
plt.xlabel('Training Cycle')
plt.ylabel('Data Complexity Level')
plt.title('Progressive Complexity Increase')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total training cycles: {len(history['loss'])}")
print(f"Best validation loss: {min(history['val_loss']):.4f}")

## Continuous Training Loop

Run this cell to train continuously and push updates every N iterations.

In [ ]:
# Continuous training loop
import time

ITERATIONS = 10  # Number of training cycles
PUSH_FREQUENCY = 3  # Push to GitHub every N iterations

for i in range(ITERATIONS):
    print(f"\n{'='*60}")
    print(f"Continuous Training Iteration {i+1}/{ITERATIONS}")
    print(f"{'='*60}\n")
    
    # Pull latest changes
    !git pull origin main
    
    # Train
    !python train.py
    
    # Push periodically
    if (i + 1) % PUSH_FREQUENCY == 0:
        commit_msg = f"Colab continuous training: iteration {i+1}/{ITERATIONS}"
        !git add model_checkpoint.pth metrics.json
        !git commit -m "{commit_msg}"
        !git push origin main
        print(f"\n✓ Pushed update {i+1}/{ITERATIONS} to GitHub")
    
    time.sleep(2)  # Brief pause between iterations

print("\n✓ Continuous training complete!")